In [ ]:
import os
import glob
from pathlib import Path
import pandas as pd
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

import scanpy as sc
import anndata

ROOT = '/path/to/original_dataset'
data_dict = {
    "dataset_name": 200 #(gene num)
}

In [ ]:
for data_name, num_genes in data_dict.items():
    data_path = os.path.join(ROOT, data_name)   # local path to dataset
    st_path = os.path.join(data_path, "st")             # ST data path

    # load ST adata
    adata_lst = []
    fn_lst = [Path(path).stem for path in glob.glob(os.path.join(st_path, "*.h5ad"))]

    first = True
    for fn in fn_lst:
        adata = anndata.read_h5ad(os.path.join(st_path, fn + ".h5ad"))
        adata_lst.append(adata)
        if first:
            common_genes = adata.var_names 
            first = False
            print(fn, adata.shape)
            continue
        common_genes = set(common_genes).intersection(set(adata.var_names))
        print(fn, adata.shape, end="\t")

    # keep common genes
    print("Length of common genes: ", len(common_genes))
    common_genes = sorted(list(common_genes))
    for fni in range(len(fn_lst)):
        adata = adata_lst[fni].copy()
        adata_lst[fni] = adata[:, common_genes].copy()
        print(fn_lst[fni], " ", adata_lst[fni].shape)
    print("Only keep common genes across the slides.")

    union_hvg = set()
    for fn_idx in range(len(fn_lst)):
        adata = adata_lst[fn_idx].copy()
        fn = fn_lst[fn_idx]
        
        sc.pp.filter_cells(adata, min_genes=1)
        sc.pp.filter_genes(adata, min_cells=1)
        sc.pp.normalize_total(adata, inplace=True)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=4096)

        union_hvg = union_hvg.union(set(adata.var_names[adata.var["highly_variable"]]))
        print(fn, len(union_hvg))

    union_hvg = sorted([gene for gene in union_hvg if not gene.startswith(("MT", "mt", "RPS", "RPL"))]) # [optional] remove mitochondrial genes and ribosomal genes
    print(len(union_hvg))

    # select union_hvg and concat all slides
    all_count_df = pd.DataFrame(adata_lst[0][:, union_hvg].X.toarray(), 
                                columns=union_hvg, 
                                index=[fn_lst[0] + "_" + str(i) for i in range(adata_lst[0].shape[0])]).T

    for fn_idx in range(1, len(fn_lst)):
        adata = adata_lst[fn_idx]
        df = pd.DataFrame(adata[:, union_hvg].X.toarray(), 
                        columns=union_hvg, 
                        index=[fn_lst[fn_idx] + "_" + str(i) for i in range(adata.shape[0])]).T
        all_count_df = pd.concat([all_count_df, df], axis=1)
        print(fn_lst[fn_idx], adata.shape, all_count_df.shape)

    all_count_df.fillna(0, inplace=True)
    all_count_df = all_count_df.T

    # order selected genes by mean and std
    all_gene_order_by_mean = all_count_df.mean(axis=0).sort_values(ascending=False).index
    all_gene_order_by_std = all_count_df.std(axis=0).sort_values(ascending=False).index

    # select top intersection of high mean and high variance genes

    selected_genes = sorted(list(set(all_gene_order_by_mean[:num_genes]).intersection(set(all_gene_order_by_std[:num_genes]))))
    print(len(selected_genes))
    gene_num = len(selected_genes)
    with open(os.path.join(ROOT + f'/{data_name}/processed_data', f"hmhvg_{gene_num}_gene_list.txt"), "w") as f:
        for gene in selected_genes:
            f.write(gene + "\n")